# Purchasability and candidate DFT validation

Keep procurement curation separate from energy screening. The existing 13-row procurement table is a raw hit list, not 13 novel systems: three reported combinations must be annotated to identify the ten retained systems. This notebook does not guess those identities.

Original notebooks and all saved outputs are preserved in `notebooks/archive/discovery_before_split_20260928/`. This reorganized notebook has not been executed.


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display

start = Path.cwd().resolve()
REPO_ROOT = next((p for p in (start, *start.parents)
                  if (p / "Code" / "model_train.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter within the CycloAddGenerator repository.")
for folder in (REPO_ROOT, REPO_ROOT / "notebooks",
               REPO_ROOT / "notebooks/04_discovery", REPO_ROOT / "notebooks/05_mechanism"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
os.chdir(REPO_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
from _workflow_paths import *
from sklearn.metrics import mean_absolute_error, r2_score


## Procurement ledger
A missing price ledger is not reconstructed from the 13 hits. A separate template may be generated; the existing hit list is preserved until a completed ledger is supplied.

In [ ]:
candidates = pd.read_csv(require_file(SCREEN_DIR / "5_DI_selected.csv"))
ledger_template = pd.concat([
    candidates[["Diene_Index", "Diene"]].rename(columns={"Diene_Index": "Index", "Diene": "Smiles"}),
    candidates[["Ene_Index", "Ene"]].rename(columns={"Ene_Index": "Index", "Ene": "Smiles"}),
], ignore_index=True).drop_duplicates()
if PURCHASE_LEDGER.is_file():
    ledger = pd.read_csv(PURCHASE_LEDGER)
    if "energy-chemical_price" not in ledger.columns:
        raise ValueError("The procurement ledger must contain energy-chemical_price.")
    prices = ledger["energy-chemical_price"]
    purchasable = ledger.loc[prices.notna() & (prices != "FALSE") & (prices != "unknown"), "Index"]
    hits = candidates.loc[candidates["Diene_Index"].isin(purchasable) & candidates["Ene_Index"].isin(purchasable)]
    hits.to_csv(SCREEN_DIR / "6_DI_selected_Purchasable.csv", index=False)
else:
    template_path = SCREEN_DIR / "5_DI_selected_Index_smiles_template.csv"
    if not template_path.exists():
        ledger_template.assign(**{"energy-chemical_price": pd.NA}).to_csv(template_path, index=False)
    print(f"Missing procurement ledger: {PURCHASE_LEDGER}. Using the existing curated hit list.")
    hits = pd.read_csv(require_file(SCREEN_DIR / "6_DI_selected_Purchasable.csv"))
display(hits)


## Candidate DFT results
Use the retained DFT/prediction table by default. Set REBUILD_DFT_JOIN to read external DFT results. Ambiguous or missing pair matches fail explicitly rather than silently selecting the first row.

In [ ]:
REBUILD_DFT_JOIN = False
if REBUILD_DFT_JOIN:
    dft = pd.read_csv(require_file(SECONDARY_ROOT / "6_DI_selected/Result_one.csv"))
    keys = ["Diene_Index", "Ene_Index"]
    cols = ["Diene_Distort", "Ene_Distort", "deltaGa(Solvent)"]
    if dft.duplicated(keys).any():
        raise ValueError("DFT pairs are not unique; resolve the intended pathway before joining.")
    validated = hits.drop(columns=cols, errors="ignore").merge(dft[keys + cols], on=keys, how="left", validate="many_to_one", indicator=True)
    if (validated["_merge"] != "both").any():
        raise ValueError("Some purchasable pairs have no DFT result.")
    validated.drop(columns="_merge", inplace=True)
    validated.to_csv(SCREEN_DIR / "6_DI_selected_Purchasable_DFT.csv", index=False)
else:
    validated = pd.read_csv(require_file(SCREEN_DIR / "6_DI_selected_Purchasable_DFT_pred.csv"))
validated["Ratio_p"] = validated["Diene_Distort_p"] / (validated["Diene_Distort_p"] + validated["Ene_Distort_p"])
validated["Ratio"] = validated["Diene_Distort"] / (validated["Diene_Distort"] + validated["Ene_Distort"])
metric_rows = []
for predicted, observed in [("deltaGa(Solvent)_p", "deltaGa(Solvent)"), ("Diene_Distort_p", "Diene_Distort"), ("Ene_Distort_p", "Ene_Distort"), ("Ratio_p", "Ratio")]:
    valid = validated[[predicted, observed]].dropna()
    metric_rows.append({"target": observed, "n": len(valid), "mae": mean_absolute_error(valid[observed], valid[predicted]), "r2": r2_score(valid[observed], valid[predicted])})
display(pd.DataFrame(metric_rows))
